# The Fokker-Planck Equation, from an SDE to its Density

Companion notebook to the note. Simulates an Ornstein-Uhlenbeck SDE, solves its
Fokker-Planck equation from scratch, and checks both against the closed-form Gaussian.

Requires only `numpy` and `matplotlib`. Runs top to bottom.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt


## 1. Simulate the SDE with Euler-Maruyama


In [ ]:
def euler_maruyama(mu, sigma, x0, T, dt, n_paths, rng):
    n_steps = int(T / dt)
    X = np.full(n_paths, float(x0))
    traj = np.empty((n_steps + 1, n_paths))
    traj[0] = X
    for k in range(n_steps):
        t = k * dt
        Z = rng.standard_normal(n_paths)
        X = X + mu(X, t) * dt + sigma(X, t) * np.sqrt(dt) * Z
        traj[k + 1] = X
    return traj

theta, sigma_ou, x0 = 1.0, 1.0, 2.0
mu  = lambda x, t: -theta * x
sig = lambda x, t: sigma_ou

dt = 1e-3
rng = np.random.default_rng(0)
traj = euler_maruyama(mu, sig, x0, T=4.0, dt=dt, n_paths=20000, rng=rng)
tgrid = np.arange(traj.shape[0]) * dt


In [ ]:
plt.figure(figsize=(8, 4.5))
for i in range(12):
    plt.plot(tgrid, traj[:, i], lw=0.9, alpha=0.8)
plt.axhline(0, color="k", lw=0.6, ls=":")
plt.xlabel("t"); plt.ylabel("$X_t$"); plt.title("OU sample paths")
plt.grid(True, alpha=0.3); plt.show()


## 2. Closed-form transition density

For OU, $X_t \sim \mathcal{N}(m(t), v(t))$ with
$m(t)=x_0 e^{-\theta t}$ and $v(t)=\frac{\sigma^2}{2\theta}(1-e^{-2\theta t})$.


In [ ]:
def analytic_pdf(x, t):
    m = x0 * np.exp(-theta * t)
    v = (sigma_ou**2 / (2*theta)) * (1 - np.exp(-2*theta*t))
    return np.exp(-(x - m)**2 / (2*v)) / np.sqrt(2*np.pi*v)


## 3. Solve the Fokker-Planck equation from scratch

Explicit finite-volume scheme in flux form: $\partial_t p = -\partial_x J$, $J = \mu p - D\,\partial_x p$, with $D=\sigma^2/2$.


In [ ]:
def solve_fokker_planck(mu_x, D, x, p0, T, dt):
    dxl = x[1] - x[0]
    xf = 0.5 * (x[:-1] + x[1:])
    muf = mu_x(xf)
    p = p0.copy()
    for _ in range(int(T / dt)):
        J = muf * 0.5 * (p[:-1] + p[1:]) - D * (p[1:] - p[:-1]) / dxl
        dpdt = np.zeros_like(p)
        dpdt[1:-1] = -(J[1:] - J[:-1]) / dxl
        p = p + dt * dpdt
        p[0] = p[-1] = 0.0
    return p

x = np.linspace(-4, 4, 801)
dx = x[1] - x[0]
p0 = np.exp(-(x - x0)**2 / (2*0.01)) / np.sqrt(2*np.pi*0.01)   # narrow spike
D = sigma_ou**2 / 2
dt_pde = 0.2 * dx**2 / D                                        # CFL stability

p_pde_1 = solve_fokker_planck(lambda z: -theta*z, D, x, p0, T=1.0, dt=dt_pde)
p_pde_4 = solve_fokker_planck(lambda z: -theta*z, D, x, p0, T=4.0, dt=dt_pde)


## 4. Three routes, one answer (t = 1)


In [ ]:
m1 = x0 * np.exp(-theta * 1.0)
v1 = (sigma_ou**2 / (2*theta)) * (1 - np.exp(-2*theta*1.0))
mc = traj[int(1.0/dt)]
pde_mean = np.sum(x * p_pde_1) * dx
pde_std  = np.sqrt(np.sum((x - pde_mean)**2 * p_pde_1) * dx)
print(f"analytic    mean={m1:.6f}  std={np.sqrt(v1):.6f}")
print(f"monte carlo mean={mc.mean():.6f}  std={mc.std():.6f}")
print(f"pde grid    mean={pde_mean:.6f}  std={pde_std:.6f}")


In [ ]:
plt.figure(figsize=(8, 4.5))
plt.hist(mc, bins=90, range=(-3, 3), density=True, color="0.8",
         edgecolor="0.6", label="Monte Carlo paths")
plt.plot(x, p_pde_1, "C0", lw=2.2, label="Fokker-Planck PDE")
plt.plot(x, analytic_pdf(x, 1.0), "C3--", lw=1.8, label="Closed-form Gaussian")
plt.xlabel("x"); plt.ylabel("p(x, 1)"); plt.xlim(-2.5, 3)
plt.title("Density at t = 1, three independent ways")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()


## 5. Stationary distribution (t = 4)


In [ ]:
stat_var = sigma_ou**2 / (2*theta)
stationary = np.exp(-x**2 / (2*stat_var)) / np.sqrt(2*np.pi*stat_var)
plt.figure(figsize=(8, 4.5))
plt.hist(traj[int(4.0/dt)], bins=90, range=(-3, 3), density=True, color="0.8",
         edgecolor="0.6", label="Monte Carlo (t = 4)")
plt.plot(x, p_pde_4, "C0", lw=2.2, label="Fokker-Planck PDE (t = 4)")
plt.plot(x, stationary, "C3--", lw=1.8, label="Stationary N(0, 1/2)")
plt.xlabel("x"); plt.ylabel("p(x)"); plt.xlim(-3, 3)
plt.title("Long-time density matches the stationary Gaussian")
plt.legend(); plt.grid(True, alpha=0.3); plt.show()
